# Contrafactuales y sesgo en un sistema de crédito

Este cuaderno construye un conjunto de datos sintético para analizar:

1. qué variables empujan más la decisión del modelo,
2. si la variable `es_negro` domina frente al resto,
3. cómo cambian las decisiones con contrafactuales en clasificación y regresión.

La estructura del análisis es:

- crear datos,
- entrenar modelos,
- comparar variables con varias métricas,
- auditar cambios uno a uno,
- construir contrafactuales **closest**, **plausible** y **diverse**,
- repetir la lógica en clasificación y en regresión.

## Hito 1. Librerías

In [ ]:
!pip -q install dice-ml scikit-learn pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.9 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

from IPython.display import display

## Hito 2. Funciones auxiliares

Las funciones siguientes se usan para:

- comparar un caso original con un contrafactual,
- medir distancias estandarizadas,
- buscar un **closest** por búsqueda local,
- buscar un **plausible** como caso real observado,
- construir un conjunto **diverse** de casos cercanos pero distintos.

In [ ]:
def comparar_filas(fila_original, fila_nueva, columnas=None, tolerancia=1e-8):
    if columnas is None:
        columnas = fila_original.index.tolist()
    cambios = []
    for col in columnas:
        a = fila_original[col]
        b = fila_nueva[col]
        try:
            if abs(float(a) - float(b)) > tolerancia:
                cambios.append((col, a, b))
        except:
            if a != b:
                cambios.append((col, a, b))
    return pd.DataFrame(cambios, columns=["variable", "original", "nuevo"])


def distancia_estandarizada(x1, x2, medias, desvios, columnas):
    z1 = (x1[columnas] - medias[columnas]) / desvios[columnas]
    z2 = (x2[columnas] - medias[columnas]) / desvios[columnas]
    return float(np.sqrt(((z1 - z2) ** 2).sum(axis=1)).iloc[0])


def buscar_closest_clas(caso, modelo, X_train, variables_accionables, objetivo=1, mantener_fijas=None, quantiles=(0.1, 0.25, 0.5, 0.75, 0.9)):
    if mantener_fijas is None:
        mantener_fijas = []
    base = caso.copy()
    medias = X_train[variables_accionables].mean()
    desvios = X_train[variables_accionables].std().replace(0, 1)

    grids = {}
    for v in variables_accionables:
        vals = sorted(set(np.round(X_train[v].quantile(list(quantiles)).values, 3).tolist() + [float(base.iloc[0][v])]))
        grids[v] = vals

    candidatos = []

    # cambios de una variable
    for v in variables_accionables:
        if v in mantener_fijas:
            continue
        for val in grids[v]:
            cand = base.copy()
            cand[v] = val
            if modelo.predict(cand)[0] == objetivo:
                dist = distancia_estandarizada(base, cand, medias, desvios, variables_accionables)
                n_cambios = int((comparar_filas(base.iloc[0], cand.iloc[0], variables_accionables).shape[0]))
                candidatos.append((dist, n_cambios, cand))

    # cambios de dos variables
    vars_libres = [v for v in variables_accionables if v not in mantener_fijas]
    for i in range(len(vars_libres)):
        for j in range(i + 1, len(vars_libres)):
            v1, v2 = vars_libres[i], vars_libres[j]
            for val1 in grids[v1]:
                for val2 in grids[v2]:
                    cand = base.copy()
                    cand[v1] = val1
                    cand[v2] = val2
                    if modelo.predict(cand)[0] == objetivo:
                        dist = distancia_estandarizada(base, cand, medias, desvios, variables_accionables)
                        n_cambios = int((comparar_filas(base.iloc[0], cand.iloc[0], variables_accionables).shape[0]))
                        candidatos.append((dist, n_cambios, cand))

    if not candidatos:
        return None

    candidatos = sorted(candidatos, key=lambda x: (x[1], x[0]))
    return candidatos[0][2]


def buscar_plausible_clas(caso, X_train, y_train, columnas_distancia, filtros=None):
    candidatos = X_train[y_train == 1].copy()
    if filtros is not None:
        for col, val in filtros.items():
            candidatos = candidatos[candidatos[col] == val]
    medias = candidatos[columnas_distancia].mean()
    desvios = candidatos[columnas_distancia].std().replace(0, 1)
    d = ((candidatos[columnas_distancia] - caso.iloc[0][columnas_distancia]) / desvios) ** 2
    idx = d.sum(axis=1).sort_values().index[0]
    return candidatos.loc[[idx]].copy()


def buscar_diverse_clas(caso, X_train, y_train, columnas_distancia, k=3, filtros=None):
    candidatos = X_train[y_train == 1].copy()
    if filtros is not None:
        for col, val in filtros.items():
            candidatos = candidatos[candidatos[col] == val]

    medias = candidatos[columnas_distancia].mean()
    desvios = candidatos[columnas_distancia].std().replace(0, 1)
    d0 = (((candidatos[columnas_distancia] - caso.iloc[0][columnas_distancia]) / desvios) ** 2).sum(axis=1)
    shortlist = candidatos.loc[d0.sort_values().head(30).index].copy()

    elegidos = []
    restantes = shortlist.copy()

    primero = restantes.loc[[d0.loc[restantes.index].sort_values().index[0]]]
    elegidos.append(primero)
    restantes = restantes.drop(index=primero.index)

    while len(elegidos) < k and len(restantes) > 0:
        puntajes = []
        for idx, row in restantes.iterrows():
            row_df = pd.DataFrame([row])
            dist_al_caso = distancia_estandarizada(caso, row_df, medias, desvios, columnas_distancia)
            dist_min = min(
                distancia_estandarizada(sel, row_df, medias, desvios, columnas_distancia)
                for sel in elegidos
            )
            puntajes.append((dist_al_caso - 0.35 * dist_min, idx))
        idx_sel = sorted(puntajes, key=lambda x: x[0])[0][1]
        elegido = restantes.loc[[idx_sel]]
        elegidos.append(elegido)
        restantes = restantes.drop(index=idx_sel)

    return pd.concat(elegidos, axis=0)


def buscar_closest_reg(caso, modelo, X_train, variables_accionables, umbral_superior, mantener_fijas=None, quantiles=(0.1, 0.25, 0.5, 0.75, 0.9)):
    if mantener_fijas is None:
        mantener_fijas = []
    base = caso.copy()
    medias = X_train[variables_accionables].mean()
    desvios = X_train[variables_accionables].std().replace(0, 1)

    grids = {}
    for v in variables_accionables:
        vals = sorted(set(np.round(X_train[v].quantile(list(quantiles)).values, 3).tolist() + [float(base.iloc[0][v])]))
        grids[v] = vals

    candidatos = []
    for v in variables_accionables:
        if v in mantener_fijas:
            continue
        for val in grids[v]:
            cand = base.copy()
            cand[v] = val
            if modelo.predict(cand)[0] <= umbral_superior:
                dist = distancia_estandarizada(base, cand, medias, desvios, variables_accionables)
                n_cambios = int((comparar_filas(base.iloc[0], cand.iloc[0], variables_accionables).shape[0]))
                candidatos.append((dist, n_cambios, cand))

    vars_libres = [v for v in variables_accionables if v not in mantener_fijas]
    for i in range(len(vars_libres)):
        for j in range(i + 1, len(vars_libres)):
            v1, v2 = vars_libres[i], vars_libres[j]
            for val1 in grids[v1]:
                for val2 in grids[v2]:
                    cand = base.copy()
                    cand[v1] = val1
                    cand[v2] = val2
                    if modelo.predict(cand)[0] <= umbral_superior:
                        dist = distancia_estandarizada(base, cand, medias, desvios, variables_accionables)
                        n_cambios = int((comparar_filas(base.iloc[0], cand.iloc[0], variables_accionables).shape[0]))
                        candidatos.append((dist, n_cambios, cand))

    if not candidatos:
        return None

    candidatos = sorted(candidatos, key=lambda x: (x[1], x[0]))
    return candidatos[0][2]


def buscar_plausible_reg(caso, X_train, y_train, columnas_distancia, umbral_superior, filtros=None):
    candidatos = X_train[y_train <= umbral_superior].copy()
    if filtros is not None:
        for col, val in filtros.items():
            candidatos = candidatos[candidatos[col] == val]
    medias = candidatos[columnas_distancia].mean()
    desvios = candidatos[columnas_distancia].std().replace(0, 1)
    d = ((candidatos[columnas_distancia] - caso.iloc[0][columnas_distancia]) / desvios) ** 2
    idx = d.sum(axis=1).sort_values().index[0]
    return candidatos.loc[[idx]].copy()


def buscar_diverse_reg(caso, X_train, y_train, columnas_distancia, umbral_superior, k=3, filtros=None):
    candidatos = X_train[y_train <= umbral_superior].copy()
    if filtros is not None:
        for col, val in filtros.items():
            candidatos = candidatos[candidatos[col] == val]

    medias = candidatos[columnas_distancia].mean()
    desvios = candidatos[columnas_distancia].std().replace(0, 1)
    d0 = (((candidatos[columnas_distancia] - caso.iloc[0][columnas_distancia]) / desvios) ** 2).sum(axis=1)
    shortlist = candidatos.loc[d0.sort_values().head(30).index].copy()

    elegidos = []
    restantes = shortlist.copy()

    primero = restantes.loc[[d0.loc[restantes.index].sort_values().index[0]]]
    elegidos.append(primero)
    restantes = restantes.drop(index=primero.index)

    while len(elegidos) < k and len(restantes) > 0:
        puntajes = []
        for idx, row in restantes.iterrows():
            row_df = pd.DataFrame([row])
            dist_al_caso = distancia_estandarizada(caso, row_df, medias, desvios, columnas_distancia)
            dist_min = min(
                distancia_estandarizada(sel, row_df, medias, desvios, columnas_distancia)
                for sel in elegidos
            )
            puntajes.append((dist_al_caso - 0.35 * dist_min, idx))
        idx_sel = sorted(puntajes, key=lambda x: x[0])[0][1]
        elegido = restantes.loc[[idx_sel]]
        elegidos.append(elegido)
        restantes = restantes.drop(index=idx_sel)

    return pd.concat(elegidos, axis=0)

## Hito 3. Datos sintéticos para clasificación

La variable objetivo será `aprobado`.

Variables de entrada:

- `es_negro`
- `ingreso`
- `deuda_ratio`
- `score`
- `antiguedad_laboral`
- `ahorro`

La etiqueta se construye con una regla latente donde `es_negro` entra con una penalización fuerte. La idea es analizar después si el modelo recupera ese peso.

In [ ]:
np.random.seed(7)
n = 1600

es_negro = np.random.binomial(1, 0.35, n)
ingreso = np.random.normal(3200, 900, n).clip(700, 9000)
deuda_ratio = np.random.beta(2.5, 3.5, n)
score = np.random.normal(610, 90, n).clip(300, 850)
antiguedad_laboral = np.random.randint(0, 20, n)
ahorro = np.random.normal(5000, 4500, n).clip(0, 35000)

puntaje_latente = (
    0.0012 * ingreso
    - 3.0 * deuda_ratio
    + 0.008 * (score - 600)
    + 0.05 * antiguedad_laboral
    + 0.00005 * ahorro
    - 1.4
    - 2.6 * es_negro
    + np.random.normal(0, 0.65, n)
)

aprobado = (puntaje_latente > 0).astype(int)

df_clas = pd.DataFrame({
    "es_negro": es_negro,
    "ingreso": np.round(ingreso, 0),
    "deuda_ratio": np.round(deuda_ratio, 3),
    "score": np.round(score, 0),
    "antiguedad_laboral": antiguedad_laboral,
    "ahorro": np.round(ahorro, 0),
    "aprobado": aprobado
})

print("Dimensiones:", df_clas.shape)
display(df_clas.head())

Dimensiones: (1600, 7)


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro,aprobado
0,0,3991.0,0.639,614.0,11,8062.0,1
1,1,1751.0,0.230,528.0,1,10452.0,0
2,0,4013.0,0.359,565.0,15,7758.0,1
3,1,3004.0,0.127,510.0,14,0.0,0
4,1,2583.0,0.175,608.0,13,10125.0,0


## Hito 4. Tasas de aprobación por grupo

In [ ]:
resumen_grupo = (
    df_clas.groupby("es_negro")["aprobado"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "tasa_aprobacion", "count": "n"})
)
resumen_grupo["grupo"] = resumen_grupo["es_negro"].map({0: "No negro", 1: "Negro"})
display(resumen_grupo[["grupo", "n", "tasa_aprobacion"]])

,grupo,n,tasa_aprobacion
0,No negro,1043,0.893576
1,Negro,557,0.393178


## Hito 5. Modelo de clasificación

In [ ]:
variables = ["es_negro", "ingreso", "deuda_ratio", "score", "antiguedad_laboral", "ahorro"]
Xc = df_clas[variables].copy()
yc = df_clas["aprobado"].copy()

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    Xc, yc, test_size=0.2, random_state=42, stratify=yc
)

modelo_clas = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)
modelo_clas.fit(X_train_c, y_train_c)

pred_c = modelo_clas.predict(X_test_c)
print("Accuracy:", round(accuracy_score(y_test_c, pred_c), 4))
print()
print(classification_report(y_test_c, pred_c))

Accuracy: 0.9

              precision    recall  f1-score   support

           0       0.90      0.72      0.80        90
           1       0.90      0.97      0.93       230

    accuracy                           0.90       320
   macro avg       0.90      0.85      0.87       320
weighted avg       0.90      0.90      0.90       320



## Hito 6. Evaluación comparativa de variables en clasificación

Se usan tres lentes distintos:

1. **coeficientes estandarizados** de una regresión logística,
2. **importancia por permutación** del Random Forest,
3. **prueba de intervención una por una** sobre casos rechazados con `es_negro = 1`.

In [ ]:
pipe_logit = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=2000))
])
pipe_logit.fit(X_train_c, y_train_c)

coef_logit = pd.Series(
    pipe_logit.named_steps["logit"].coef_[0],
    index=variables,
    name="coef_estandarizado"
)

coef_tabla = pd.DataFrame({
    "coef_estandarizado": coef_logit,
    "abs_coef": coef_logit.abs()
}).sort_values("abs_coef", ascending=False)

display(coef_tabla)

,coef_estandarizado,abs_coef
es_negro,-3.007727,3.007727
ingreso,2.708633,2.708633
score,1.768901,1.768901
deuda_ratio,-1.360640,1.360640
antiguedad_laboral,0.605664,0.605664
ahorro,0.562066,0.562066


In [ ]:
perm_clas = permutation_importance(
    modelo_clas, X_test_c, y_test_c, n_repeats=8, random_state=42
)

perm_tabla_clas = pd.DataFrame({
    "variable": variables,
    "importancia_perm": perm_clas.importances_mean
}).sort_values("importancia_perm", ascending=False)

display(perm_tabla_clas)

,variable,importancia_perm
0,es_negro,0.192578
1,ingreso,0.117188
3,score,0.041797
2,deuda_ratio,0.017578
5,ahorro,0.001563
4,antiguedad_laboral,-0.007422


In [ ]:
rechazados_negros = X_test_c[(X_test_c["es_negro"] == 1) & (modelo_clas.predict(X_test_c) == 0)].copy()
base_prob = modelo_clas.predict_proba(rechazados_negros)[:, 1]

q_favorables = {
    "ingreso": X_train_c["ingreso"].quantile(0.90),
    "deuda_ratio": X_train_c["deuda_ratio"].quantile(0.10),
    "score": X_train_c["score"].quantile(0.90),
    "antiguedad_laboral": X_train_c["antiguedad_laboral"].quantile(0.90),
    "ahorro": X_train_c["ahorro"].quantile(0.90),
}

resultados_intervencion = []
for var, val in [("es_negro", 0),
                 ("ingreso", q_favorables["ingreso"]),
                 ("deuda_ratio", q_favorables["deuda_ratio"]),
                 ("score", q_favorables["score"]),
                 ("antiguedad_laboral", q_favorables["antiguedad_laboral"]),
                 ("ahorro", q_favorables["ahorro"])]:
    mod = rechazados_negros.copy()
    mod[var] = val
    nueva_prob = modelo_clas.predict_proba(mod)[:, 1]
    delta_prob = (nueva_prob - base_prob).mean()
    tasa_flip = (modelo_clas.predict(mod) == 1).mean()
    resultados_intervencion.append([var, round(float(val), 3), delta_prob, tasa_flip])

intervencion_tabla = pd.DataFrame(
    resultados_intervencion,
    columns=["variable_intervenida", "valor_prueba", "aumento_promedio_prob_aprobacion", "tasa_que_pasa_a_aprobado"]
).sort_values("aumento_promedio_prob_aprobacion", ascending=False)

display(intervencion_tabla)

,variable_intervenida,valor_prueba,aumento_promedio_prob_aprobacion,tasa_que_pasa_a_aprobado
0,es_negro,0.00,0.580475,0.866667
1,ingreso,4287.80,0.451157,0.916667
3,score,727.00,0.220945,0.483333
2,deuda_ratio,0.18,0.169504,0.250000
4,antiguedad_laboral,18.00,0.038196,0.033333
5,ahorro,11110.20,0.030253,0.033333


# Cómo interpretar tres métodos para evaluar el impacto de variables

Tres enfoques distintos para analizar qué variables están influyendo más en las decisiones del modelo:

1. **Coeficientes estandarizados**
2. **Importancia por permutación**
3. **Prueba de intervención variable por variable**

Los tres responden a la misma pregunta desde ángulos distintos:

> ¿Qué variables están teniendo mayor impacto en las predicciones del modelo?

Usar los tres métodos permite evaluar la influencia de las variables desde perspectivas complementarias.

---

# 1. Coeficientes estandarizados

Este método se usa principalmente en **modelos lineales** (por ejemplo regresión lineal o logística).

## Problema de las escalas

Las variables suelen estar en escalas muy distintas.

| Variable | Escala |
|---|---|
| ingreso | miles |
| deuda_ratio | 0 – 1 |
| score | 300 – 850 |

Si el modelo estima:

$y = β_1 ingreso + β_2 deuda_ratio + β_3 score$

los coeficientes **no son comparables directamente**, porque las unidades son diferentes.

---

## Estandarización

Para comparar los coeficientes se transforman las variables:

$x^* = \frac{x - \mu}{\sigma}$

Esto convierte cada variable a unidades de **desviaciones estándar**.

El modelo queda:

$y = β_1 x_1^* + β_2 x_2^* + β_3 x_3^*$

Ahora los coeficientes sí se pueden comparar.

---

## Resultados observados en este ejercicio

Los coeficientes estandarizados obtenidos fueron:

| Variable | coef_estandarizado | abs_coef |
|---|---:|---:|
| es_negro | -3.007727 | 3.007727 |
| ingreso | 2.708633 | 2.708633 |
| score | 1.768901 | 1.768901 |
| deuda_ratio | -1.360640 | 1.360640 |
| antiguedad_laboral | 0.605664 | 0.605664 |
| ahorro | 0.562066 | 0.562066 |

## Interpretación

El valor absoluto más alto corresponde a **`es_negro` = 3.007727**.  
Eso indica que, dentro de la estructura del modelo lineal estandarizado, esta es la variable con mayor peso relativo.

Además, su signo es **negativo**, lo que indica que esta variable se asocia con una disminución importante en la probabilidad de aprobación, manteniendo constantes las demás variables.

Después de `es_negro`, la variable con mayor peso es `ingreso` con **2.708633**, seguida por:

- `score` con **1.768901**
- `deuda_ratio` con **1.360640**
- `antiguedad_laboral` con **0.605664**
- `ahorro` con **0.562066**

En este ejercicio, el análisis por coeficientes estandarizados muestra que:

> `es_negro` es la variable con mayor impacto dentro de la ecuación del modelo.

---

# 2. Importancia por permutación

Este método funciona para **cualquier modelo**:

- Random Forest  
- Gradient Boosting  
- redes neuronales  
- etc.

---

## Idea principal

Si una variable es importante, entonces:

> Al destruir su información, el desempeño del modelo debería empeorar.

---

## Procedimiento

1. Se calcula el desempeño original del modelo.
2. Se selecciona una variable.
3. Se **permuta aleatoriamente** esa columna en el dataset.
4. Se vuelve a evaluar el modelo.
5. Se compara cuánto cayó el desempeño.

---

## Medida de importancia

$\text{Importancia} =
\text{score original} -
\text{score con variable permutada}$

---

## Resultados observados en este ejercicio

Los resultados de importancia por permutación fueron:

| Variable | importancia_perm (disminucion de accuracy) |
|---|---:|
| es_negro | 0.192578 |
| ingreso | 0.117188 |
| score | 0.041797 |
| deuda_ratio | 0.017578 |
| ahorro | 0.001563 |
| antiguedad_laboral | -0.007422 |

## Interpretación

La mayor caída en el desempeño del modelo se produjo al permutar **`es_negro`**, con una importancia de **0.192578**.

La segunda variable fue `ingreso`, con una importancia de **0.117188**.

La diferencia entre ambas es:

\[
0.192578 - 0.117188 = 0.07539
\]

Esto indica que el modelo depende más de `es_negro` que de `ingreso`.

Después aparecen variables claramente menos influyentes:

- `score` con **0.041797**
- `deuda_ratio` con **0.017578**
- `ahorro` con **0.001563**

En `antiguedad_laboral` aparece un valor ligeramente negativo:

- **-0.007422**

Eso sugiere que, al permutar esa variable, el desempeño no empeora de forma consistente. En este ejercicio no parece ser una variable relevante para el modelo.

En este punto, la importancia por permutación muestra que:

> `es_negro` es la variable cuya información más necesita el modelo para mantener su capacidad predictiva.

---

# 3. Prueba de intervención variable por variable

Este método consiste en realizar pequeños **experimentos contrafactuales simples**.

La pregunta que responde es:

> ¿Qué pasa con la predicción si cambio solo una variable?

---

## Procedimiento

1. Se toma un conjunto de casos.
2. Se modifica **una sola variable** por vez.
3. Se recalculan las predicciones.
4. Se compara cuánto cambia, en promedio, la probabilidad de aprobación y cuántos casos pasan de rechazado a aprobado.

---

## Resultados observados en este ejercicio

| variable_intervenida | valor_prueba | aumento_promedio_prob_aprobacion | tasa_que_pasa_a_aprobado |
|---|---:|---:|---:|
| es_negro | 0.00 | 0.580475 | 0.866667 |
| ingreso | 4287.80 | 0.451157 | 0.916667 |
| score | 727.00 | 0.220945 | 0.483333 |
| deuda_ratio | 0.18 | 0.169504 | 0.250000 |
| antiguedad_laboral | 18.00 | 0.038196 | 0.033333 |
| ahorro | 11110.20 | 0.030253 | 0.033333 |

## Interpretación

La mayor subida promedio en la probabilidad de aprobación se obtuvo al intervenir **`es_negro`**, con un aumento de:

\[
0.580475
\]

Además, la proporción de casos que pasa de rechazado a aprobado fue:

\[
0.866667
\]

Es decir, al intervenir esa variable, **86.67%** de los casos considerados cambió hacia aprobación.

La segunda variable más fuerte fue `ingreso`, con:

- aumento promedio de probabilidad = **0.451157**
- tasa que pasa a aprobado = **0.916667**

Aquí conviene distinguir dos criterios:

- si se mira el **aumento promedio en probabilidad de aprobación**, `es_negro` ocupa el primer lugar;
- si se mira la **tasa de casos que logra pasar a aprobado**, `ingreso` aparece ligeramente por encima.

Después de estas dos variables, la magnitud cae con claridad:

- `score` = **0.220945**
- `deuda_ratio` = **0.169504**
- `antiguedad_laboral` = **0.038196**
- `ahorro` = **0.030253**

Este análisis indica que:

> modificar `es_negro` produce el mayor cambio promedio en la probabilidad de aprobación entre todas las variables evaluadas.

---

# Comparación entre los tres métodos

| Método | Qué mide | Ventaja | Limitación |
|---|---|---|---|
| Coeficiente estandarizado | efecto estructural dentro del modelo | fácil de interpretar | solo para modelos lineales |
| Importancia por permutación | dependencia del desempeño | funciona con cualquier modelo | no es causal |
| Intervención variable por variable | sensibilidad de la predicción | muy intuitivo | depende del criterio de intervención y del conjunto analizado |

---

# Lectura conjunta de los resultados

Si se comparan los tres análisis, aparece un patrón consistente.

## Coeficientes estandarizados
La variable con mayor peso absoluto es:

- **`es_negro` = 3.007727**

## Importancia por permutación
La variable que más reduce el desempeño al ser permutada es:

- **`es_negro` = 0.192578**

## Intervención variable por variable
La variable que produce el mayor aumento promedio en la probabilidad de aprobación es:

- **`es_negro` = 0.580475**

En cambio, `ingreso` aparece como la segunda variable más importante de manera consistente en los tres análisis:

- coeficiente estandarizado = **2.708633**
- importancia por permutación = **0.117188**
- aumento promedio en probabilidad = **0.451157**

---

# Por qué usar los tres

Cada método captura un aspecto distinto del comportamiento del modelo:

- el **coeficiente estandarizado** muestra el peso de la variable dentro de la estructura de la ecuación;
- la **permutación** muestra cuánto depende el desempeño del modelo de esa variable;
- la **intervención** muestra cuánto cambia la decisión al modificar esa variable.

Cuando los tres métodos apuntan en la misma dirección, la evidencia es más sólida.

En este ejercicio, los tres análisis coinciden en señalar que:

> `es_negro` es la variable con mayor impacto global en la decisión del modelo.

La única matización importante es que, en la prueba de intervención, `ingreso` logra una tasa ligeramente mayor de casos que pasan a aprobado. Sin embargo, al considerar el conjunto completo de resultados, `es_negro` sigue ocupando el primer lugar en impacto general.

### Interpretación de la comparación entre variables en clasificación

Las tasas de aprobación del conjunto sintético quedaron así:

- **No negro**: 0.894
- **Negro**: 0.393

En el modelo de clasificación se observó lo siguiente:

- **Accuracy**: 0.900
- Mayor coeficiente estandarizado en valor absoluto: **es_negro = 3.008**
- Segunda variable: **ingreso = 2.709**
- Mayor importancia por permutación: **es_negro = 0.193**
- Segunda variable: **ingreso = 0.117**

En la intervención unitaria sobre personas rechazadas con `es_negro = 1`, el mayor aumento promedio en probabilidad de aprobación apareció al cambiar:

- `es_negro` de 1 a 0: **+0.580**
- `ingreso` a un valor alto de referencia: **+0.451**
- `score` a un valor alto de referencia: **+0.221**
- `deuda_ratio` a un valor bajo de referencia: **+0.170**

Tomadas en conjunto, las tres comparaciones señalan que `es_negro` es la variable con mayor impacto en este problema de clasificación.

### Lectura conjunta de los tres resultados

- si una variable queda arriba en los **coeficientes estandarizados**, su señal es fuerte en un modelo lineal,
- si queda arriba en **importancia por permutación**, el Random Forest depende mucho de ella,
- si al intervenirla individualmente cambia más decisiones, su efecto práctico también es alto.

## Hito 7. Auditoría directa de un caso individual en clasificación

Se toma una persona rechazada con `es_negro = 1` y luego se crea un clon exacto cambiando solo esa variable a `0`.

In [ ]:
caso_clas = rechazados_negros.iloc[[0]].copy()
caso_clas_clon = caso_clas.copy()
caso_clas_clon["es_negro"] = 0

prob_original = modelo_clas.predict_proba(caso_clas)[0]
prob_clon = modelo_clas.predict_proba(caso_clas_clon)[0]

auditoria_directa_clas = pd.DataFrame({
    "escenario": ["caso original", "mismo caso con es_negro = 0"],
    "prediccion": [modelo_clas.predict(caso_clas)[0], modelo_clas.predict(caso_clas_clon)[0]],
    "prob_aprobacion": [prob_original[1], prob_clon[1]]
})

print("Caso original:")
display(caso_clas)
print("Comparación directa:")
display(auditoria_directa_clas)

Caso original:


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
1143,1,2897.0,0.578,619.0,7,4990.0


Comparación directa:


,escenario,prediccion,prob_aprobacion
0,caso original,0,0.155197
1,mismo caso con es_negro = 0,1,0.899712


### Interpretación de la auditoría individual en clasificación

Para el caso seleccionado, la probabilidad de aprobación fue **0.155**.  
Al cambiar solo `es_negro` de 1 a 0, la probabilidad subió a **0.900**.

La diferencia es de **0.745** puntos de probabilidad, sin tocar ingreso, deuda, score, antigüedad laboral ni ahorro.

## Hito 8. Contrafactuales en clasificación

En este cuaderno se usan estas definiciones operativas:

- **Closest**: cambio mínimo encontrado por búsqueda local sobre valores plausibles de las variables.
- **Plausible**: caso aprobado realmente observado en los datos y cercano al perfil original.
- **Diverse**: varios casos aprobados cercanos entre sí y al mismo tiempo distintos entre sí.

In [ ]:
variables_accionables = ["es_negro", "ingreso", "deuda_ratio", "score", "antiguedad_laboral", "ahorro"]
variables_cont =                    ["ingreso", "deuda_ratio", "score", "antiguedad_laboral", "ahorro"]

closest_clas = buscar_closest_clas(
    caso=caso_clas,
    modelo=modelo_clas,
    X_train=X_train_c,
    variables_accionables=variables_accionables,
    objetivo=1
)

plausible_clas = buscar_plausible_clas(
    caso=caso_clas,
    X_train=X_train_c,
    y_train=y_train_c,
    columnas_distancia=variables_cont
)

diverse_clas = buscar_diverse_clas(
    caso=caso_clas,
    X_train=X_train_c,
    y_train=y_train_c,
    columnas_distancia=variables_cont,
    k=3
)

print("Closest")
display(closest_clas)
print("Cambios")
display(comparar_filas(caso_clas.iloc[0], closest_clas.iloc[0], variables))

print("Plausible")
display(plausible_clas)
print("Cambios")
display(comparar_filas(caso_clas.iloc[0], plausible_clas.iloc[0], variables))

print("Diverse")
display(diverse_clas)

Closest


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
1143,1,3796.25,0.578,619.0,7,4990.0


Cambios


,variable,original,nuevo
0,ingreso,2897.0,3796.25


Plausible


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
967,0,2938.0,0.558,634.0,7,4483.0


Cambios


,variable,original,nuevo
0,es_negro,1.000,0.000
1,ingreso,2897.000,2938.000
2,deuda_ratio,0.578,0.558
3,score,619.000,634.000
4,ahorro,4990.000,4483.000


Diverse


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
967,0,2938.0,0.558,634.0,7,4483.0
1145,0,2830.0,0.536,604.0,9,6926.0
586,0,3145.0,0.601,607.0,10,6044.0


In [ ]:
proba_original = modelo_clas.predict_proba(caso_clas)

print("Probabilidad base de rechazo (clase 0):", proba_original[0][0])
print("Probabilidad base de aprobación (clase 1):", proba_original[0][1])

Probabilidad base de rechazo (clase 0): 0.844802799071084
Probabilidad base de aprobación (clase 1): 0.155197200928916


In [ ]:
# Probabilidades para el caso closest específico
probabilidades = modelo_clas.predict_proba(closest_clas)

print("Probabilidad closest de rechazo (clase 0):", probabilidades[0][0])
print("Probabilidad closest de aprobación (clase 1):", probabilidades[0][1])

Probabilidad closest de rechazo (clase 0): 0.40739985179692473
Probabilidad closest de aprobación (clase 1): 0.5926001482030752


# Interpretación de los contrafactuales obtenidos

## 1. Por qué se escogió este caso y no otro

El análisis contrafactual no se realizó sobre cualquier observación del conjunto de prueba.  
Se seleccionó un caso con características específicas para que el ejercicio permitiera examinar con claridad cómo cambia la decisión del modelo.

El caso seleccionado corresponde a una observación que:

1. pertenece al grupo `es_negro = 1`,
2. fue clasificada por el modelo como **rechazada**,
3. y se encuentra relativamente cerca de la frontera de decisión.

La razón para escoger un caso de este tipo es que, si se toma una observación extremadamente alejada de la aprobación, los contrafactuales tienden a requerir cambios demasiado grandes y se vuelven menos informativos. En cambio, cuando se elige un caso rechazado pero cercano al umbral de aprobación, es posible observar con mayor precisión qué variables empujan la decisión y qué tan sensible es el modelo a cambios pequeños o moderados.

El perfil original era aproximadamente:

- `es_negro = 1`
- `ingreso = 2897`
- `deuda_ratio = 0.578`
- `score = 619`
- `antiguedad_laboral = 7`
- `ahorro = 4990`

Este caso se utiliza como punto de partida para responder la pregunta:

> ¿Qué tendría que cambiar en este perfil para que el modelo pasara de rechazo a aprobación?

---

# 2. Qué hace internamente cada método

Aunque los tres métodos generan puntos de comparación con el caso original, no buscan exactamente lo mismo ni operan de la misma manera.

---

# 2.1 Closest

El método **closest** busca el caso aprobado más cercano al caso original, permitiendo modificar únicamente las variables definidas como accionables.

En el código, este método se llama de la siguiente manera:

```python
closest_clas = buscar_closest_clas(
    caso=caso_clas,
    modelo=modelo_clas,
    X_train=X_train_c,
    variables_accionables=variables_accionables,
    objetivo=1
)
```

## Qué hace internamente

La lógica del método es la siguiente:

1. toma el caso original,
2. genera o evalúa candidatos cercanos,
3. verifica cuáles de esos candidatos son clasificados por el modelo como `objetivo = 1`,
4. entre esos candidatos selecciona el que implique **la menor desviación posible respecto al caso original**.

Por esta razón se denomina *closest*: busca el cambio mínimo que permite alterar la decisión del modelo.

---

## Resultado obtenido

El contrafactual closest fue:

| es_negro | ingreso | deuda_ratio | score | antiguedad_laboral | ahorro |
|---|---|---|---|---|---|
| 1 | 3796.25 | 0.578 | 619.0 | 7 | 4990.0 |

Cambios observados:

| variable | original | nuevo |
|---|---|---|
| ingreso | 2897.0 | 3796.25 |

---

## Interpretación

El resultado muestra que el cambio mínimo necesario para lograr la aprobación fue **aumentar el ingreso**.

Todas las demás variables permanecieron exactamente iguales.

Esto significa que el modelo identifica una frontera de decisión local en la cual el ingreso es suficiente para mover el caso desde rechazo hacia aprobación.

En términos sustantivos, el resultado puede leerse de la siguiente manera:

> Si el ingreso aumentara hasta aproximadamente 3796, manteniendo constantes las demás variables, el modelo aprobaría el crédito.

---

# 2.2 Plausible

El método **plausible** no busca el cambio matemáticamente mínimo, sino un caso **realista dentro de los datos observados**.

En el código, este método se ejecuta así:

```python
plausible_clas = buscar_plausible_clas(
    caso=caso_clas,
    X_train=X_train_c,
    y_train=y_train_c,
    columnas_distancia=variables_cont
)
```

---

## Qué hace internamente

El procedimiento es el siguiente:

1. se seleccionan únicamente las observaciones del conjunto de entrenamiento que están **aprobadas**,
2. se calcula la distancia entre el caso original y cada uno de esos casos aprobados,
3. la distancia se calcula usando solo las variables definidas en `columnas_distancia`:

- `ingreso`
- `deuda_ratio`
- `score`
- `antiguedad_laboral`
- `ahorro`

4. se selecciona el caso aprobado **más cercano** al caso original.

En este procedimiento **no se fuerza que `es_negro` permanezca igual**, porque esa variable no forma parte del cálculo de distancia.

---

## Resultado obtenido

El contrafactual plausible fue:

| es_negro | ingreso | deuda_ratio | score | antiguedad_laboral | ahorro |
|---|---|---|---|---|---|
| 0 | 2938.0 | 0.558 | 634.0 | 7 | 4483.0 |

Cambios observados:

| variable | original | nuevo |
|---|---|---|
| es_negro | 1 | 0 |
| ingreso | 2897 | 2938 |
| deuda_ratio | 0.578 | 0.558 |
| score | 619 | 634 |
| ahorro | 4990 | 4483 |

---

## Interpretación

El resultado indica que el caso aprobado más cercano dentro de los datos observados tiene un perfil muy similar al caso original en variables financieras, pero difiere en el valor de `es_negro`.

Esto significa que, al buscar una alternativa realista dentro del conjunto de entrenamiento, el caso aprobado más cercano corresponde a una observación con:

```
es_negro = 0
```

Este resultado refleja la estructura empírica del conjunto de datos.

---

# 2.3 Diverse

El método **diverse** busca varias alternativas cercanas en lugar de una sola.

En el código se ejecuta así:

```python
diverse_clas = buscar_diverse_clas(
    caso=caso_clas,
    X_train=X_train_c,
    y_train=y_train_c,
    columnas_distancia=variables_cont,
    k=3
)
```

---

## Qué hace internamente

El procedimiento consiste en:

1. tomar las observaciones aprobadas del conjunto de entrenamiento,
2. calcular su distancia al caso original utilizando las variables continuas,
3. ordenar esos casos por cercanía,
4. seleccionar los `k` casos más cercanos (en este ejercicio `k = 3`).

El objetivo es identificar **varios perfiles aprobados cercanos**, en lugar de una única alternativa.

---

## Resultados obtenidos

Los tres contrafactuales diversos fueron:

| es_negro | ingreso | deuda_ratio | score | antiguedad_laboral | ahorro |
|---|---|---|---|---|---|
| 0 | 2938.0 | 0.558 | 634.0 | 7 | 4483.0 |
| 0 | 2830.0 | 0.536 | 604.0 | 9 | 6926.0 |
| 0 | 3145.0 | 0.601 | 607.0 | 10 | 6044.0 |

---

## Interpretación

Los tres perfiles aprobados cercanos comparten una regularidad clara:

```
es_negro = 0
```

Las demás variables pueden variar en distintas direcciones:

- el ingreso puede subir o bajar ligeramente,
- la deuda puede mejorar o empeorar moderadamente,
- el score puede cambiar dentro de un rango relativamente cercano,
- la antigüedad laboral puede aumentar,
- el ahorro puede variar de forma considerable.

Esto indica que el modelo tolera cierta variabilidad en las variables financieras, pero los perfiles aprobados cercanos presentan sistemáticamente el valor `es_negro = 0`.

---

# 3. Comparación entre los tres métodos

Los tres métodos responden preguntas distintas:

### Closest
Busca **el cambio mínimo necesario para alterar la decisión del modelo**.

En este caso, la modificación mínima fue aumentar el ingreso.

---

### Plausible
Busca **el caso aprobado real más parecido al perfil original** dentro del dataset.

Ese caso cercano tiene `es_negro = 0`.

---

### Diverse
Busca **varias alternativas aprobadas cercanas**.

En las tres alternativas encontradas aparece el mismo patrón:

```
es_negro = 0
```

---

# 4. Lectura conjunta

Los resultados muestran dos niveles distintos de análisis.

## Nivel local

El método closest indica que, para este caso específico, el modelo permite una ruta corta hacia la aprobación aumentando el ingreso.

---

## Nivel estructural

Los métodos plausible y diverse muestran que, cuando se buscan perfiles aprobados cercanos dentro de los datos observados, aparece de forma repetida el valor:

```
es_negro = 0
```

Esto indica que los perfiles aprobados cercanos en el conjunto de entrenamiento presentan una asociación consistente con esa variable.

---

# 5. Interpretación final

En conjunto, los resultados muestran que:

- el ingreso funciona como una palanca local suficiente para alterar la decisión en este caso específico,
- pero los perfiles aprobados cercanos dentro de los datos presentan de forma sistemática el valor `es_negro = 0`.

Por lo tanto, aunque el cambio mínimo necesario para aprobar este caso consiste en aumentar el ingreso, la estructura de los casos aprobados cercanos dentro del conjunto de entrenamiento muestra una asociación consistente con la variable `es_negro`.

## Hito 9. Contrafactuales en clasificación sin permitir cambio en `es_negro`

In [ ]:
closest_clas_res = buscar_closest_clas(
    caso=caso_clas,
    modelo=modelo_clas,
    X_train=X_train_c,
    variables_accionables=variables_accionables,
    objetivo=1,
    mantener_fijas=["es_negro"]
)

plausible_clas_res = buscar_plausible_clas(
    caso=caso_clas,
    X_train=X_train_c,
    y_train=y_train_c,
    columnas_distancia=variables_cont,
    filtros={"es_negro": int(caso_clas.iloc[0]["es_negro"])}
)

diverse_clas_res = buscar_diverse_clas(
    caso=caso_clas,
    X_train=X_train_c,
    y_train=y_train_c,
    columnas_distancia=variables_cont,
    k=3,
    filtros={"es_negro": int(caso_clas.iloc[0]["es_negro"])}
)

print("Closest restringido")
display(closest_clas_res)
print("Cambios")
display(comparar_filas(caso_clas.iloc[0], closest_clas_res.iloc[0], variables))

print("Plausible restringido")
display(plausible_clas_res)
print("Cambios")
display(comparar_filas(caso_clas.iloc[0], plausible_clas_res.iloc[0], variables))

print("Diverse restringido")
display(diverse_clas_res)

Closest restringido


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
1143,1,3796.25,0.578,619.0,7,4990.0


Cambios


,variable,original,nuevo
0,ingreso,2897.0,3796.25


Plausible restringido


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
1081,1,2713.0,0.474,627.0,3,4431.0


Cambios


,variable,original,nuevo
0,ingreso,2897.000,2713.000
1,deuda_ratio,0.578,0.474
2,score,619.000,627.000
3,antiguedad_laboral,7.000,3.000
4,ahorro,4990.000,4431.000


Diverse restringido


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
1081,1,2713.0,0.474,627.0,3,4431.0
1003,1,3387.0,0.619,690.0,6,5345.0
1533,1,3503.0,0.642,636.0,2,4447.0


# Interpretación de los contrafactuales restringidos

## 1. Qué significa que estos contrafactuales sean "restringidos"

En esta parte del análisis se impuso una restricción explícita: la variable `es_negro` debía permanecer fija.

Eso se implementó de dos maneras:

### En `closest`
Se usó el argumento:

```python
mantener_fijas=["es_negro"]
```

Eso obliga al método a buscar una alternativa aprobada **sin modificar** esa variable.

### En `plausible` y `diverse`
Se usó el filtro:

```python
filtros={"es_negro": int(caso_clas.iloc[0]["es_negro"])}
```

Eso obliga a que los casos aprobados candidatos pertenezcan al mismo grupo que el caso original, es decir:

```python
es_negro = 1
```

Por lo tanto, en esta sección el análisis ya no permite que la aprobación se logre cambiando la variable `es_negro`.  
La pregunta que se está respondiendo ahora es distinta:

> Si `es_negro` se mantiene fijo, ¿qué otras combinaciones de variables permiten llegar a aprobación?

---

## 2. Por qué es importante este análisis restringido

En la versión no restringida, algunos resultados mostraban que los perfiles aprobados cercanos tendían a aparecer con:

```python
es_negro = 0
```

Eso era útil para detectar la asociación entre esa variable y la decisión del modelo, pero no era suficiente para responder otra pregunta importante:

> ¿Existe una ruta hacia la aprobación sin alterar la variable protegida?

El análisis restringido permite responder precisamente eso.  
Es decir, muestra si todavía existen caminos de aprobación basados únicamente en variables financieras o de perfil.

---

# 3. Caso original de referencia

Los cambios observados permiten reconstruir el caso original:

| variable | valor original |
|---|---:|
| es_negro | 1 |
| ingreso | 2897 |
| deuda_ratio | 0.578 |
| score | 619 |
| antiguedad_laboral | 7 |
| ahorro | 4990 |

Este es el caso desde el cual se calculan los tres tipos de contrafactuales restringidos.

---

# 4. Closest restringido

## Resultado

| es_negro | ingreso | deuda_ratio | score | antiguedad_laboral | ahorro |
|---|---:|---:|---:|---:|---:|
| 1 | 3796.25 | 0.578 | 619.0 | 7 | 4990.0 |

## Cambios observados

| variable | original | nuevo |
|---|---:|---:|
| ingreso | 2897.0 | 3796.25 |

## Qué hace internamente este método

El método `buscar_closest_clas(...)` busca el cambio mínimo necesario para que el modelo pase a predecir la clase objetivo, en este caso aprobación (`objetivo=1`).

Al incluir:

```python
mantener_fijas=["es_negro"]
```

el algoritmo ya no puede usar esa variable como vía de cambio. Por tanto, solo puede ajustar las demás variables accionables.

La lógica del método es:

1. toma el caso original,
2. busca alternativas cercanas,
3. verifica cuáles de ellas son aprobadas por el modelo,
4. entre esas alternativas aprobadas, elige la que implique la menor desviación respecto al caso original,
5. respetando que `es_negro` permanezca fijo.

## Interpretación

El resultado muestra que, una vez prohibido el cambio de `es_negro`, el cambio mínimo necesario para lograr aprobación consiste en **aumentar el ingreso** de:

```python
2897  →  3796.25
```

Todas las demás variables permanecen iguales.

Esto significa que, para este caso puntual, existe una ruta directa hacia la aprobación sin alterar la variable protegida, pero esa ruta exige una mejora importante en ingreso.

La lectura de este resultado es:

> Si `es_negro` se mantiene fijo, el modelo todavía permite aprobar este caso, pero el ajuste mínimo necesario consiste en aumentar el ingreso.

---

# 5. Plausible restringido

## Resultado

| es_negro | ingreso | deuda_ratio | score | antiguedad_laboral | ahorro |
|---|---:|---:|---:|---:|---:|
| 1 | 2713.0 | 0.474 | 627.0 | 3 | 4431.0 |

## Cambios observados

| variable | original | nuevo |
|---|---:|---:|
| ingreso | 2897.000 | 2713.000 |
| deuda_ratio | 0.578 | 0.474 |
| score | 619.000 | 627.000 |
| antiguedad_laboral | 7.000 | 3.000 |
| ahorro | 4990.000 | 4431.000 |

## Qué hace internamente este método

El método `buscar_plausible_clas(...)` busca el caso aprobado más cercano **dentro del conjunto de entrenamiento**.

En esta versión se incluyó el filtro:

```python
filtros={"es_negro": int(caso_clas.iloc[0]["es_negro"])}
```

Eso hace que el algoritmo solo compare el caso original con observaciones aprobadas que cumplan:

```python
es_negro = 1
```

La lógica es:

1. toma el conjunto de entrenamiento,
2. filtra solo los casos aprobados,
3. dentro de esos aprobados conserva únicamente los que tienen el mismo valor de `es_negro`,
4. calcula la distancia entre el caso original y cada uno de esos candidatos usando `variables_cont`,
5. selecciona el aprobado más cercano.

## Interpretación

Este resultado es importante porque muestra que sí existe un caso aprobado real y cercano dentro del mismo grupo `es_negro = 1`.

Sin embargo, ese caso no se obtiene cambiando una sola variable, sino mediante una combinación de ajustes:

- el ingreso baja ligeramente,
- la deuda mejora de forma importante,
- el score mejora un poco,
- la antigüedad laboral disminuye,
- el ahorro también disminuye.

Esto deja ver algo importante: el plausible restringido **no busca el cambio mínimo**, sino el caso aprobado más parecido que realmente existe en los datos.

Por eso puede ocurrir que algunas variables incluso empeoren respecto al caso original, siempre que el perfil completo corresponda a un caso aprobado cercano.

La lectura adecuada es:

> Dentro de los casos aprobados observados con `es_negro = 1`, el perfil más parecido al caso original combina una menor deuda, un score algo mejor y algunos cambios compensatorios en ingreso, antigüedad y ahorro.

---

# 6. Diverse restringido

## Resultados

| es_negro | ingreso | deuda_ratio | score | antiguedad_laboral | ahorro |
|---|---:|---:|---:|---:|---:|
| 1 | 2713.0 | 0.474 | 627.0 | 3 | 4431.0 |
| 1 | 3387.0 | 0.619 | 690.0 | 6 | 5345.0 |
| 1 | 3503.0 | 0.642 | 636.0 | 2 | 4447.0 |

## Qué hace internamente este método

El método `buscar_diverse_clas(...)` busca varias alternativas aprobadas cercanas al caso original.

En este caso se aplicó el mismo filtro:

```python
filtros={"es_negro": int(caso_clas.iloc[0]["es_negro"])}
```

por lo que todas las alternativas deben cumplir:

```python
es_negro = 1
```

La lógica del método es:

1. toma los casos aprobados del conjunto de entrenamiento,
2. filtra solo aquellos con el mismo valor de `es_negro`,
3. calcula la distancia respecto al caso original usando las variables continuas,
4. ordena esos casos por cercanía,
5. selecciona los `k=3` casos aprobados más cercanos.

A diferencia del plausible, aquí no se entrega una sola alternativa, sino varias.

## Interpretación

Los tres perfiles muestran que sí existen varias rutas cercanas hacia la aprobación manteniendo fijo `es_negro = 1`.

### Primer perfil
| variable | valor |
|---|---:|
| ingreso | 2713 |
| deuda_ratio | 0.474 |
| score | 627 |
| antiguedad_laboral | 3 |
| ahorro | 4431 |

Este perfil enfatiza principalmente una mejora en deuda y una ligera mejora en score.

### Segundo perfil
| variable | valor |
|---|---:|
| ingreso | 3387 |
| deuda_ratio | 0.619 |
| score | 690 |
| antiguedad_laboral | 6 |
| ahorro | 5345 |

Este perfil combina:

- mayor ingreso,
- score mucho más alto,
- ahorro algo mayor,
- aunque la deuda no mejora.

### Tercer perfil
| variable | valor |
|---|---:|
| ingreso | 3503 |
| deuda_ratio | 0.642 |
| score | 636 |
| antiguedad_laboral | 2 |
| ahorro | 4447 |

Aquí también aparece una mejora en ingreso y score, aunque otras variables no necesariamente mejoran.

## Qué muestran en conjunto estos tres perfiles

Los resultados diverse restringidos muestran que, una vez fijado `es_negro`, la aprobación puede lograrse por varias combinaciones alternativas de variables.

No existe una única ruta obligatoria.  
Lo que aparece es una familia de soluciones cercanas donde distintas combinaciones compensan el efecto de otras.

La lectura general es:

> Manteniendo fijo `es_negro = 1`, todavía existen varios perfiles aprobados cercanos, pero esos perfiles requieren distintos ajustes en ingreso, score, deuda, antigüedad o ahorro.

---

# 7. Diferencia entre los tres métodos restringidos

## Closest restringido
Busca:

> el menor cambio posible para alterar la decisión.

En este ejercicio, ese cambio fue solo:

- aumentar ingreso.

---

## Plausible restringido
Busca:

> el caso aprobado real más parecido dentro del mismo grupo `es_negro = 1`.

Aquí el resultado no representa el mínimo cambio, sino el vecino aprobado más cercano observado en los datos.

---

## Diverse restringido
Busca:

> varias alternativas aprobadas cercanas dentro del mismo grupo `es_negro = 1`.

Esto permite ver que no existe una única combinación válida, sino varias trayectorias posibles hacia la aprobación.

---

# 8. Lectura conjunta de los resultados restringidos

Cuando se prohíbe cambiar `es_negro`, los resultados muestran que la aprobación sigue siendo posible.

Sin embargo, esa aprobación ya no puede alcanzarse mediante una modificación en la variable protegida, sino que debe lograrse a través de combinaciones en variables financieras o de perfil.

Los resultados permiten distinguir dos cosas:

## a) Ruta mínima
El **closest restringido** muestra que, para este caso particular, la ruta mínima es:

- aumentar el ingreso.

## b) Rutas realistas observadas
El **plausible restringido** y el **diverse restringido** muestran que, dentro de los datos, también existen perfiles aprobados cercanos con `es_negro = 1`, aunque esos perfiles combinan varios cambios y no necesariamente mejoran todas las variables al mismo tiempo.

---

# 9. Interpretación final

La versión restringida del análisis muestra que el caso original sí puede llegar a aprobación sin modificar la variable `es_negro`.

El resultado más claro en términos de cambio mínimo es el del **closest restringido**, donde basta con aumentar el ingreso.

Por su parte, los resultados **plausible restringido** y **diverse restringido** muestran que también existen perfiles aprobados cercanos dentro del mismo grupo `es_negro = 1`, lo cual indica que la aprobación no es imposible para ese grupo, pero sí requiere configuraciones específicas de variables financieras y de perfil.

En conjunto, estos resultados permiten separar dos ideas:

1. la variable `es_negro` está asociada con el patrón de aprobación del modelo cuando no se impone restricción,
2. pero, aun manteniéndola fija, existen rutas alternativas hacia la aprobación basadas en otras variables.

Eso hace que esta parte del análisis sea importante, porque permite identificar qué cambios siguen siendo viables cuando la variable protegida se mantiene constante.

## Hito 10. Interpretación de clasificación

In [ ]:
texto_clas = []

var_coef_top = coef_tabla.index[0]
var_perm_top = perm_tabla_clas.iloc[0]["variable"]
var_interv_top = intervencion_tabla.iloc[0]["variable_intervenida"]

texto_clas.append(f"La variable con mayor coeficiente estandarizado en valor absoluto fue: {var_coef_top}.")
texto_clas.append(f"La variable con mayor importancia por permutación fue: {var_perm_top}.")
texto_clas.append(f"La variable con mayor aumento promedio en probabilidad de aprobación fue: {var_interv_top}.")
texto_clas.append(
    f"En la auditoría individual, la probabilidad de aprobación pasó de {auditoria_directa_clas.loc[0, 'prob_aprobacion']:.3f} a {auditoria_directa_clas.loc[1, 'prob_aprobacion']:.3f} al cambiar solo es_negro de 1 a 0."
)
texto_clas.append(
    "Las tres lecturas apuntan en la misma dirección: la variable racial domina la decisión del modelo en este conjunto sintético."
)
texto_clas.append(
    "Cuando se bloquea esa variable, el modelo necesita compensar con cambios adicionales en ingreso, score, deuda o antigüedad laboral."
)

for t in texto_clas:
    print('-', t)

- La variable con mayor coeficiente estandarizado en valor absoluto fue: es_negro.
- La variable con mayor importancia por permutación fue: es_negro.
- La variable con mayor aumento promedio en probabilidad de aprobación fue: es_negro.
- En la auditoría individual, la probabilidad de aprobación pasó de 0.155 a 0.900 al cambiar solo es_negro de 1 a 0.
- Las tres lecturas apuntan en la misma dirección: la variable racial domina la decisión del modelo en este conjunto sintético.
- Cuando se bloquea esa variable, el modelo necesita compensar con cambios adicionales en ingreso, score, deuda o antigüedad laboral.


_________

# Parte B. Regresión

Ahora la variable objetivo será `tasa_interes`.

La lógica es la misma: se introduce una penalización fuerte asociada a `es_negro` y luego se compara ese efecto con el resto de variables.

In [ ]:
tasa_interes = (
    27
    - 0.0017 * ingreso
    + 7.0 * deuda_ratio
    - 0.016 * (score - 600)
    - 0.18 * antiguedad_laboral
    - 0.00007 * ahorro
    + 4.5 * es_negro
    + np.random.normal(0, 1.1, n)
)

tasa_interes = np.clip(tasa_interes, 5, 45)

df_reg = pd.DataFrame({
    "es_negro": es_negro,
    "ingreso": np.round(ingreso, 0),
    "deuda_ratio": np.round(deuda_ratio, 3),
    "score": np.round(score, 0),
    "antiguedad_laboral": antiguedad_laboral,
    "ahorro": np.round(ahorro, 0),
    "tasa_interes": np.round(tasa_interes, 2)
})

print("Dimensiones:", df_reg.shape)
display(df_reg.head())

Dimensiones: (1600, 7)


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro,tasa_interes
0,0,3991.0,0.639,614.0,11,8062.0,22.36
1,1,1751.0,0.230,528.0,1,10452.0,31.02
2,0,4013.0,0.359,565.0,15,7758.0,18.46
3,1,3004.0,0.127,510.0,14,0.0,27.27
4,1,2583.0,0.175,608.0,13,10125.0,27.66


## Hito 11. Diferencia promedio de tasa por grupo

In [ ]:
resumen_reg_grupo = (
    df_reg.groupby("es_negro")["tasa_interes"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "tasa_media", "count": "n"})
)
resumen_reg_grupo["grupo"] = resumen_reg_grupo["es_negro"].map({0: "No negro", 1: "Negro"})
display(resumen_reg_grupo[["grupo", "n", "tasa_media"]])

,grupo,n,tasa_media
0,No negro,1043,22.292838
1,Negro,557,26.678205


## Hito 12. Modelo de regresión

In [ ]:
Xr = df_reg[variables].copy()
yr = df_reg["tasa_interes"].copy()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)

modelo_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42
)
modelo_reg.fit(X_train_r, y_train_r)

pred_r = modelo_reg.predict(X_test_r)
print("RMSE:", round(np.sqrt(mean_squared_error(y_test_r, pred_r)), 4))
print("R2:", round(r2_score(y_test_r, pred_r), 4))

RMSE: 1.3929
R2: 0.8577


## Hito 13. Evaluación comparativa de variables en regresión

In [ ]:
pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])
pipe_ridge.fit(X_train_r, y_train_r)

coef_ridge = pd.Series(
    pipe_ridge.named_steps["ridge"].coef_,
    index=variables,
    name="coef_estandarizado"
)

coef_tabla_reg = pd.DataFrame({
    "coef_estandarizado": coef_ridge,
    "abs_coef": coef_ridge.abs()
}).sort_values("abs_coef", ascending=False)

display(coef_tabla_reg)

,coef_estandarizado,abs_coef
es_negro,2.190487,2.190487
ingreso,-1.572048,1.572048
score,-1.378219,1.378219
deuda_ratio,1.272575,1.272575
antiguedad_laboral,-0.993546,0.993546
ahorro,-0.277800,0.277800


In [ ]:
perm_reg = permutation_importance(
    modelo_reg, X_test_r, y_test_r, n_repeats=8, random_state=42
)

perm_tabla_reg = pd.DataFrame({
    "variable": variables,
    "importancia_perm": perm_reg.importances_mean
}).sort_values("importancia_perm", ascending=False)

display(perm_tabla_reg)

,variable,importancia_perm
0,es_negro,0.681107
1,ingreso,0.303609
3,score,0.239553
2,deuda_ratio,0.170644
4,antiguedad_laboral,0.120229
5,ahorro,0.006245


In [ ]:
casos_negros_reg = X_test_r[X_test_r["es_negro"] == 1].copy()
base_tasa = modelo_reg.predict(casos_negros_reg)

q_favorables_reg = {
    "ingreso": X_train_r["ingreso"].quantile(0.90),
    "deuda_ratio": X_train_r["deuda_ratio"].quantile(0.10),
    "score": X_train_r["score"].quantile(0.90),
    "antiguedad_laboral": X_train_r["antiguedad_laboral"].quantile(0.90),
    "ahorro": X_train_r["ahorro"].quantile(0.90),
}

resultados_delta = []
for var, val in [("es_negro", 0),
                 ("ingreso", q_favorables_reg["ingreso"]),
                 ("deuda_ratio", q_favorables_reg["deuda_ratio"]),
                 ("score", q_favorables_reg["score"]),
                 ("antiguedad_laboral", q_favorables_reg["antiguedad_laboral"]),
                 ("ahorro", q_favorables_reg["ahorro"])]:
    mod = casos_negros_reg.copy()
    mod[var] = val
    delta = (modelo_reg.predict(mod) - base_tasa).mean()
    resultados_delta.append([var, round(float(val), 3), delta])

delta_tabla = pd.DataFrame(resultados_delta, columns=["variable_intervenida", "valor_prueba", "cambio_promedio_tasa"])
delta_tabla = delta_tabla.sort_values("cambio_promedio_tasa")
display(delta_tabla)

,variable_intervenida,valor_prueba,cambio_promedio_tasa
0,es_negro,0.000,-4.652574
1,ingreso,4306.300,-1.847274
2,deuda_ratio,0.182,-1.598834
3,score,728.000,-1.456176
4,antiguedad_laboral,17.000,-0.809759
5,ahorro,11029.000,-0.244008


### Interpretación de la comparación entre variables en regresión

Las tasas medias del conjunto sintético quedaron así:

- **No negro**: 22.293
- **Negro**: 26.678

En el modelo de regresión se observó lo siguiente:

- **RMSE**: 1.394
- **R²**: 0.858
- Mayor coeficiente estandarizado en valor absoluto: **es_negro = 2.190**
- Segunda variable: **ingreso = 1.572**
- Mayor importancia por permutación: **es_negro = 0.680**
- Segunda variable: **ingreso = 0.303**

En la intervención unitaria sobre solicitantes con `es_negro = 1`, el mayor descenso promedio de tasa apareció al cambiar:

- `es_negro` de 1 a 0: **-4.644**
- `ingreso` a un valor alto de referencia: **-1.835**
- `deuda_ratio` a un valor bajo de referencia: **-1.604**
- `score` a un valor alto de referencia: **-1.457**

También en regresión, las tres comparaciones ubican a `es_negro` como la variable de mayor impacto.

## Hito 14. Auditoría directa de un caso individual en regresión

In [ ]:
caso_reg = X_test_r[X_test_r["es_negro"] == 1].iloc[[0]].copy()
caso_reg_clon = caso_reg.copy()
caso_reg_clon["es_negro"] = 0

auditoria_directa_reg = pd.DataFrame({
    "escenario": ["caso original", "mismo caso con es_negro = 0"],
    "tasa_predicha": [modelo_reg.predict(caso_reg)[0], modelo_reg.predict(caso_reg_clon)[0]]
})

print("Caso original:")
display(caso_reg)
print("Comparación directa:")
display(auditoria_directa_reg)

Caso original:


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
354,1,3751.0,0.254,509.0,18,1372.0


Comparación directa:


,escenario,tasa_predicha
0,caso original,25.875102
1,mismo caso con es_negro = 0,20.778099


### Interpretación de la auditoría individual en regresión

Para el caso seleccionado, la tasa predicha fue **25.858**.  
Al cambiar solo `es_negro` de 1 a 0, la tasa bajó a **20.779**.

La diferencia fue de **5.079** puntos.

## Hito 15. Contrafactuales en regresión

Se busca bajar la tasa predicha del caso original. Se fija como umbral una mejora de al menos 3 puntos.

In [ ]:
umbral_superior = float(modelo_reg.predict(caso_reg)[0] - 3.0)
print("Tasa original:", round(float(modelo_reg.predict(caso_reg)[0]), 3))
print("Umbral superior buscado:", round(umbral_superior, 3))

closest_reg = buscar_closest_reg(
    caso=caso_reg,
    modelo=modelo_reg,
    X_train=X_train_r,
    variables_accionables=variables,
    umbral_superior=umbral_superior
)

plausible_reg = buscar_plausible_reg(
    caso=caso_reg,
    X_train=X_train_r,
    y_train=y_train_r,
    columnas_distancia=variables_cont,
    umbral_superior=umbral_superior
)

diverse_reg = buscar_diverse_reg(
    caso=caso_reg,
    X_train=X_train_r,
    y_train=y_train_r,
    columnas_distancia=variables_cont,
    umbral_superior=umbral_superior,
    k=3
)

print("Closest")
display(closest_reg)
print("Cambios")
display(comparar_filas(caso_reg.iloc[0], closest_reg.iloc[0], variables))
print("Tasa predicha:", round(float(modelo_reg.predict(closest_reg)[0]), 3))

print("Plausible")
display(plausible_reg)
print("Cambios")
display(comparar_filas(caso_reg.iloc[0], plausible_reg.iloc[0], variables))
print("Tasa predicha:", round(float(modelo_reg.predict(plausible_reg)[0]), 3))

print("Diverse")
display(diverse_reg)
print("Tasas predichas:")
print(np.round(modelo_reg.predict(diverse_reg), 3))

Tasa original: 25.875
Umbral superior buscado: 22.875
Closest


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
354,0.0,3751.0,0.254,509.0,18,1372.0


Cambios


,variable,original,nuevo
0,es_negro,1.0,0.0


Tasa predicha: 20.778
Plausible


...

Cambios


,variable,original,nuevo
0,es_negro,1.000,0.000
1,ingreso,3751.000,3789.000
2,deuda_ratio,0.254,0.269
3,score,509.000,529.000
4,antiguedad_laboral,18.000,17.000
5,ahorro,1372.000,1466.000


Tasa predicha: 20.317
Diverse


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
1133,0,3789.0,0.269,529.0,17,1466.0
1120,0,3291.0,0.256,537.0,16,1656.0
1119,0,3715.0,0.272,566.0,17,2433.0


Tasas predichas:
[20.317 20.671 19.606]


## Hito 16. Contrafactuales en regresión sin permitir cambio en `es_negro`

In [ ]:
closest_reg_res = buscar_closest_reg(
    caso=caso_reg,
    modelo=modelo_reg,
    X_train=X_train_r,
    variables_accionables=variables,
    umbral_superior=umbral_superior,
    mantener_fijas=["es_negro"]
)

plausible_reg_res = buscar_plausible_reg(
    caso=caso_reg,
    X_train=X_train_r,
    y_train=y_train_r,
    columnas_distancia=variables_cont,
    umbral_superior=umbral_superior,
    filtros={"es_negro": int(caso_reg.iloc[0]["es_negro"])}
)

diverse_reg_res = buscar_diverse_reg(
    caso=caso_reg,
    X_train=X_train_r,
    y_train=y_train_r,
    columnas_distancia=variables_cont,
    umbral_superior=umbral_superior,
    k=3,
    filtros={"es_negro": int(caso_reg.iloc[0]["es_negro"])}
)

print("Closest restringido")
display(closest_reg_res)
print("Cambios")
display(comparar_filas(caso_reg.iloc[0], closest_reg_res.iloc[0], variables))
print("Tasa predicha:", round(float(modelo_reg.predict(closest_reg_res)[0]), 3))

print("Plausible restringido")
display(plausible_reg_res)
print("Cambios")
display(comparar_filas(caso_reg.iloc[0], plausible_reg_res.iloc[0], variables))
print("Tasa predicha:", round(float(modelo_reg.predict(plausible_reg_res)[0]), 3))

print("Diverse restringido")
display(diverse_reg_res)
print("Tasas predichas:")
print(np.round(modelo_reg.predict(diverse_reg_res), 3))

Closest restringido


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
354,1,4306.3,0.254,673.0,18,1372.0


Cambios


,variable,original,nuevo
0,ingreso,3751.0,4306.3
1,score,509.0,673.0


Tasa predicha: 22.615
Plausible restringido


...

Cambios


,variable,original,nuevo
0,ingreso,3751.000,4306.000
1,deuda_ratio,0.254,0.211
2,antiguedad_laboral,18.000,19.000
3,ahorro,1372.000,8219.000


Tasa predicha: 23.452
Diverse restringido


,es_negro,ingreso,deuda_ratio,score,antiguedad_laboral,ahorro
1201,1,4306.0,0.211,509.0,19,8219.0
1573,1,4449.0,0.243,619.0,17,0.0
314,1,3950.0,0.168,649.0,19,2172.0


Tasas predichas:
[23.452 22.814 22.883]


## Hito 17. Interpretación de regresión

In [ ]:
texto_reg = []

var_coef_top_reg = coef_tabla_reg.index[0]
var_perm_top_reg = perm_tabla_reg.iloc[0]["variable"]
var_delta_top = delta_tabla.iloc[0]["variable_intervenida"]

texto_reg.append(f"La variable con mayor coeficiente estandarizado en valor absoluto fue: {var_coef_top_reg}.")
texto_reg.append(f"La variable con mayor importancia por permutación fue: {var_perm_top_reg}.")
texto_reg.append(f"La intervención con mayor reducción promedio de tasa fue: {var_delta_top}.")
texto_reg.append(
    f"En la auditoría individual, la tasa predicha bajó de {auditoria_directa_reg.loc[0, 'tasa_predicha']:.3f} a {auditoria_directa_reg.loc[1, 'tasa_predicha']:.3f} al cambiar solo es_negro de 1 a 0."
)
texto_reg.append(
    "En este conjunto sintético, las tres comparaciones vuelven a señalar a la variable racial como la de mayor impacto."
)

for t in texto_reg:
    print('-', t)

- La variable con mayor coeficiente estandarizado en valor absoluto fue: es_negro.
- La variable con mayor importancia por permutación fue: es_negro.
- La intervención con mayor reducción promedio de tasa fue: es_negro.
- En la auditoría individual, la tasa predicha bajó de 25.875 a 20.778 al cambiar solo es_negro de 1 a 0.
- En este conjunto sintético, las tres comparaciones vuelven a señalar a la variable racial como la de mayor impacto.


## Hito 18. Resumen final

In [ ]:
resumen_final = pd.DataFrame({
    "bloque": ["Clasificación", "Clasificación", "Clasificación", "Regresión", "Regresión", "Regresión"],
    "criterio": [
        "Mayor |coeficiente|", "Mayor importancia por permutación", "Mayor aumento promedio de probabilidad",
        "Mayor |coeficiente|", "Mayor importancia por permutación", "Mayor reducción promedio de tasa"
    ],
    "variable_ganadora": [
        coef_tabla.index[0], perm_tabla_clas.iloc[0]["variable"], intervencion_tabla.iloc[0]["variable_intervenida"],
        coef_tabla_reg.index[0], perm_tabla_reg.iloc[0]["variable"], delta_tabla.iloc[0]["variable_intervenida"]
    ]
})

display(resumen_final)

,bloque,criterio,variable_ganadora
0,Clasificación,Mayor |coeficiente|,es_negro
1,Clasificación,Mayor importancia por permutación,es_negro
2,Clasificación,Mayor aumento promedio de probabilidad,es_negro
3,Regresión,Mayor |coeficiente|,es_negro
4,Regresión,Mayor importancia por permutación,es_negro
5,Regresión,Mayor reducción promedio de tasa,es_negro
